# Experiment 40 — Backward-free SparseWalker with hard negatives

Surgical extension of Experiment 39. Same corrected SparseWalker v1.1 recurrence and same local learning rules; only the negative curriculum changes.

- epochs 1–7: 32 random negatives, matching the clean bootstrap used before
- epoch 8+: 8 model-mined hard + 16 random + 8 popularity negatives
- hard candidates come from in-batch next items + popular items + a fresh random pool
- **no warm start, no optimizer, no backward, no autograd learning**
- `last.pt` is saved every epoch; rerun with `RESUME=True` after a Colab crash


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, runpy, json, torch
from pathlib import Path
REPO='/content/Sparsewalker'
BRANCH='agent/local-contrastive-hard-negatives-v2'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
for p in [f'{REPO}/src',f'{REPO}/experiments',f'{REPO}/benchmarks']:
    if p not in sys.path: sys.path.insert(0,p)
import sparsewalker
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'torch',torch.__version__)
print('BRANCH',BRANCH,'PACKAGE',sparsewalker.__file__)


## Run / resume

Set `RESUME=True` if the runtime dies. Because there is no optimizer state, the saved model + epoch are enough to continue the deterministic epoch-indexed training loop.


In [ ]:
RESUME=False
SCRIPT=f'{REPO}/experiments/run_amazon_local_contrastive_hard_negatives.py'
argv=[SCRIPT,'--dataset','beauty','--epochs','70','--batch-size','512','--eval-batch-size','1024','--hard-start-epoch','8','--hard-negatives','8','--random-negatives','16','--popular-negatives','8','--random-pool','256','--popular-pool','256']
if RESUME: argv.append('--resume')
sys.argv=argv
runpy.run_path(SCRIPT,run_name='__main__')


## Inspect trajectory

The important diagnostics are:
- `val_NDCG@10`: must beat Experiment 39's **0.040516**
- `mean_hard_similarity` vs `mean_random_similarity`: verifies that mining is actually finding harder mistakes
- `mean_negative_prob`: should stop collapsing into an easy-negative regime
- `hard_mining_active_fraction`: should jump from 0 to 1 at epoch 8


In [ ]:
import pandas as pd
root=Path('/content/drive/MyDrive/sparsewalker_local_contrastive_hard/beauty/seed42')
hp=root/'history.json'
if hp.exists():
    h=pd.DataFrame(json.loads(hp.read_text()))
    cols=['epoch','mean_contrastive_margin','mean_positive_prob','mean_negative_prob','mean_hard_similarity','mean_random_similarity','mean_candidate_pool_size','hard_mining_active_fraction','val_NDCG@10','val_HR@10','positions_per_s']
    display(h[[c for c in cols if c in h.columns]])
    if len(h):
        best=h.loc[h['val_NDCG@10'].idxmax()]
        print('BEST',best.to_dict())
        print('LC_V1_BEST_VAL_NDCG',0.040515991131704246)
        print('SASREC_VAL_NDCG',0.04296780165590764)
else:
    print('No history yet.')


## Crash recovery / final result

If the notebook crashes after training, reopen it, run setup, set `RESUME=True`, and run the training cell. If epoch 70 was already saved, it will immediately load the best checkpoint and emit `LCH_RESULT`.


In [ ]:
rp=root/'result.json'
bp=root/'best.pt'
lp=root/'last.pt'
print('best.pt',bp.exists(),'last.pt',lp.exists(),'result.json',rp.exists())
if rp.exists():
    print(json.dumps(json.loads(rp.read_text()),indent=2))
elif lp.exists():
    ck=torch.load(lp,map_location='cpu')
    print('RECOVERABLE_FROM_EPOCH',ck['epoch'],'BEST_EPOCH',ck.get('best_epoch'),'BEST_VAL',ck.get('best'))
